In [3]:
import os
import cv2

# ========================= CONFIGURATION =========================
IMAGE_DIR = r"D:\internship\internship_computer_visions_engineering\data_Set\validation\images"
CLASS_ID = 0  # The class ID for your bounding box
# =================================================================

# Global variables to track mouse state and coordinates
drawing = False
ix, iy = -1, -1
x1, y1, x2, y2 = -1, -1, -1, -1

def draw_bbox(event, x, y, flags, param):
    global ix, iy, x1, y1, x2, y2, drawing

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        ix, iy = x, y
        x1, y1, x2, y2 = x, y, x, y

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            x2, y2 = x, y

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        x2, y2 = x, y

def main():
    global ix, iy, x1, y1, x2, y2, drawing
    
    if not os.path.exists(IMAGE_DIR):
        print(f"Error: The directory '{IMAGE_DIR}' does not exist. Check your path.")
        return

    parent_dir = os.path.dirname(IMAGE_DIR)
    labels_dir = os.path.join(parent_dir, "labels")
    os.makedirs(labels_dir, exist_ok=True)
    
    valid_exts = (".png", ".jpg", ".jpeg", ".bmp")
    all_images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_exts)])

    # Filter to find ONLY images that are missing labels OR have empty labels (backgrounds)
    target_images = []
    for img_name in all_images:
        base_no_ext = os.path.splitext(img_name)[0]
        txt_path = os.path.join(labels_dir, f"{base_no_ext}.txt")
        
        # Scenario 1: Label doesn't exist
        if not os.path.exists(txt_path):
            target_images.append(img_name)
        # Scenario 2: Label exists but is empty (0 bytes / background)
        elif os.path.getsize(txt_path) == 0:
            target_images.append(img_name)

    if not target_images:
        print("Success! No empty background labels or missing labels found in this split.")
        return

    print("\n" + "="*60)
    print(f" Found {len(target_images)} background/unlabeled images to annotate.")
    print(" DIRECT GUI KEYBOARD CONTROLS (Use inside Image Window):")
    print("  - [Drag Mouse] : Draw bounding box")
    print("  - [c]          : CLEAR/RESET the drawn box")
    print("  - [d]          : DELETE image and label from disk")
    print("  - [Space/Enter]: SAVE box & NEXT (Skips saving empty if no box is drawn)")
    print("  - [s]          : SKIP this image (keep it as background/empty)")
    print("  - [Esc]        : QUIT")
    print("="*60 + "\n")

    cv2.namedWindow("Annotation Window")
    cv2.setMouseCallback("Annotation Window", draw_bbox)

    idx = 0
    while idx < len(target_images):
        img_name = target_images[idx]
        img_path = os.path.join(IMAGE_DIR, img_name)
        
        base_no_ext = os.path.splitext(img_name)[0]
        txt_path = os.path.join(labels_dir, f"{base_no_ext}.txt")

        if not os.path.exists(img_path):
            idx += 1
            continue

        frame = cv2.imread(img_path)
        if frame is None:
            idx += 1
            continue

        img_h, img_w, _ = frame.shape
        x1, y1, x2, y2 = -1, -1, -1, -1
        skip_image = False

        while True:
            temp_img = frame.copy()

            # Draw target bounding box
            if x1 != -1 and y1 != -1 and x2 != -1 and y2 != -1:
                cv2.rectangle(temp_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

            # Inform user how many backgrounds are left
            cv2.putText(temp_img, f"Remaining backgrounds to review: {len(target_images) - idx}", 
                        (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.imshow("Annotation Window", temp_img)
            key = cv2.waitKey(10) & 0xFF

            # --- KEY BINDINGS ---

            # Clear box
            if key == ord('c'):
                x1, y1, x2, y2 = -1, -1, -1, -1
                print("Cleared box.")

            # 'd' key: DELETE current image and empty label
            elif key == ord('d'):
                try:
                    cv2.destroyAllWindows()  # Temporarily close OpenCV window for terminal input
                    confirm = input(f"Are you sure you want to delete {img_name}? (y/n): ").lower().strip()
                    if confirm == 'y':
                        if os.path.exists(img_path):
                            os.remove(img_path)
                        if os.path.exists(txt_path):
                            os.remove(txt_path)
                        print(f"Deleted {img_name} from disk.")
                        idx += 1
                        skip_image = True
                        
                        # Re-initialize GUI window context
                        cv2.namedWindow("Annotation Window")
                        cv2.setMouseCallback("Annotation Window", draw_bbox)
                        break
                    else:
                        print("Delete canceled.")
                        # Recreate window
                        cv2.namedWindow("Annotation Window")
                        cv2.setMouseCallback("Annotation Window", draw_bbox)
                except Exception as e:
                    print(f"Error deleting file: {e}")

            # Skip button 's': leaves the file as an empty background label and goes to next
            elif key == ord('s'):
                print(f"Skipped (Kept as background): {img_name}")
                idx += 1
                skip_image = True
                break

            # Space (32) or Enter (13): SAVE and proceed
            elif key == 32 or key == 13:
                try:
                    if x1 != -1 and x2 != -1:
                        xmin = min(x1, x2)
                        ymin = min(y1, y2)
                        xmax = max(x1, x2)
                        ymax = max(y1, y2)
                        
                        # Normalize values
                        box_w = xmax - xmin
                        box_h = ymax - ymin
                        x_center = xmin + (box_w / 2.0)
                        y_center = ymin + (box_h / 2.0)
                        
                        norm_cx = min(max(x_center / img_w, 0.0), 1.0)
                        norm_cy = min(max(y_center / img_h, 0.0), 1.0)
                        norm_w = min(max(box_w / img_w, 0.0), 1.0)
                        norm_h = min(max(box_h / img_h, 0.0), 1.0)
                        
                        # Write standard 5-column format
                        with open(txt_path, 'w') as f:
                            f.write(f"{CLASS_ID} {norm_cx:.6f} {norm_cy:.6f} {norm_w:.6f} {norm_h:.6f}\n")
                        print(f"Saved: {os.path.basename(txt_path)}")
                        idx += 1
                        skip_image = True
                        break
                    else:
                        print("No box drawn! Draw a box, press 's' to skip, or 'd' to delete.")
                except Exception as e:
                    print(f"Error saving: {e}")

            # Esc key (27): QUIT
            elif key == 27:
                print("Exiting...")
                cv2.destroyAllWindows()
                return

        if skip_image:
            continue

    cv2.destroyAllWindows()
    print("\n--- Review Complete! ---")


if __name__ == "__main__":
    main()


 Found 73 background/unlabeled images to annotate.
 DIRECT GUI KEYBOARD CONTROLS (Use inside Image Window):
  - [Drag Mouse] : Draw bounding box
  - [c]          : CLEAR/RESET the drawn box
  - [d]          : DELETE image and label from disk
  - [Space/Enter]: SAVE box & NEXT (Skips saving empty if no box is drawn)
  - [s]          : SKIP this image (keep it as background/empty)
  - [Esc]        : QUIT

Deleted 02305154375.jpg from disk.
Saved: 02635059922.txt
Saved: 02741151662.txt
Saved: 03005451152.txt
Saved: 03034083844.txt
Saved: 03041128475.txt
Saved: 03130720000.txt
Saved: 03374694903.txt
Saved: 03497088793.txt
Saved: 03575177344.txt
Saved: 03589133176.txt
Saved: 04078347392.txt
Saved: 04180110001.txt
Saved: 04404064844.txt
Saved: 04447200009.txt
Saved: 04605745316.txt
Saved: 04791520002.txt
Saved: 05012645015.txt
Saved: 05018274661.txt
Saved: 05164210006.txt
Saved: 05170659089.txt
Saved: 05363200006.txt
Saved: 05397555698.txt
Cleared box.
Saved: 05405396762.txt
Saved: 05777877